In [ ]:
# Dependencies
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error

# ARIMA forecast extraction
def extract_arima_forecasts(group):
    arima_feats = {}
    group = group.copy()
    group["date"] = pd.to_datetime(group["date"])

    for label in ["income", "expense", "surplus"]:
        try:
            if label == "income":
                series = group[group["type"].str.lower() == "credit"].set_index("date")["amount"].resample("M").sum().fillna(0)
            elif label == "expense":
                series = group[group["type"].str.lower() == "debit"].set_index("date")["amount"].abs().resample("M").sum().fillna(0)
            elif label == "surplus":
                income_series = group[group["type"].str.lower() == "credit"].set_index("date")["amount"].resample("M").sum().fillna(0)
                expense_series = group[group["type"].str.lower() == "debit"].set_index("date")["amount"].abs().resample("M").sum().fillna(0)
                series = (income_series - expense_series).fillna(0)

            if len(series) >= 6:
                model = ARIMA(series, order=(1, 1, 1))
                model_fit = model.fit()
                forecast = model_fit.forecast(steps=1)
                arima_feats[f"forecast_{label}"] = forecast.iloc[0]
                arima_feats[f"forecast_error_{label}"] = model_fit.resid.std()
            else:
                arima_feats[f"forecast_{label}"] = np.nan
                arima_feats[f"forecast_error_{label}"] = np.nan
        except Exception as e:
            print(f"ARIMA fallback for {label}: {e}")
            arima_feats[f"forecast_{label}"] = np.nan
            arima_feats[f"forecast_error_{label}"] = np.nan

    return arima_feats

# feature Engineering
def feature_engineering(transactions):
    features = []
    transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
    transactions["amount"] = pd.to_numeric(transactions["amount"], errors="coerce").fillna(0.0)

    for user_id, group in transactions.groupby("userId"):
        user_feat = {"userId": user_id}
        group['month'] = group['date'].dt.to_period('M')

        monthly_income = group[group["type"].str.lower() == "credit"].groupby("month")["amount"].sum().fillna(0)
        monthly_expense = group[group["type"].str.lower() == "debit"].groupby("month")["amount"].apply(lambda x: x.abs().sum()).fillna(0)

        surplus = monthly_income - monthly_expense
        total_income = monthly_income.sum()
        total_expense = monthly_expense.sum()

        user_feat.update({
            "surplus_ratio": surplus.mean() / (total_income + 1e-6),
            "income_volatility": monthly_income.std() / (monthly_income.mean() + 1e-6),
            "min_monthly_income": (monthly_income.min() / (total_income + 1e-6)) if total_income > 0 else 0,
            "gambling_ratio": group[group["predicted_subcategory"].str.lower() == "gambling"]["amount"].abs().sum() / (total_expense + 1e-6),
            "fixed_cost_ratio": group[(group["is_recurring"]) & (group["type"].str.lower() == "debit")]["amount"].abs().sum() / (total_expense + 1e-6),
            "net_negative_months_ratio": (surplus < 0).sum() / (len(surplus) + 1e-6),
            "max_monthly_expense_ratio": (monthly_expense.max() / (total_expense + 1e-6)) if total_expense > 0 else 0,
            "emergency_buffer_ratio": surplus.mean() / (monthly_expense.mean() + 1e-6) if monthly_expense.mean() > 0 else 0,
            "recurring_income_ratio": group[(group["is_recurring"]) & (group["recurrence_type"] == "income")]["amount"].sum() / (total_income + 1e-6),
            "surplus_frequency_ratio": (surplus > 0).sum() / (len(surplus) + 1e-6)
        })

        user_feat.update(extract_arima_forecasts(group))
        features.append(user_feat)

    print(f"Feature engineering completed for {len(features)} users.")
    return pd.DataFrame(features)

# Forecasting evaluation
def evaluate_arima_performance(transactions, forecast_horizon=3):
    results = {"income": [], "expense": [], "surplus": []}
    transactions["date"] = pd.to_datetime(transactions["date"])

    for user_id, group in transactions.groupby("userId"):
        group = group.sort_values("date")

        for label in ["income", "expense", "surplus"]:
            try:
                if label == "income":
                    series = group[group["type"].str.lower() == "credit"].set_index("date")["amount"].resample("M").sum().fillna(0)
                elif label == "expense":
                    series = group[group["type"].str.lower() == "debit"].set_index("date")["amount"].abs().resample("M").sum().fillna(0)
                elif label == "surplus":
                    income_series = group[group["type"].str.lower() == "credit"].set_index("date")["amount"].resample("M").sum().fillna(0)
                    expense_series = group[group["type"].str.lower() == "debit"].set_index("date")["amount"].abs().resample("M").sum().fillna(0)
                    series = (income_series - expense_series).fillna(0)

                if len(series) >= (forecast_horizon + 6):
                    train = series[:-forecast_horizon]
                    test = series[-forecast_horizon:]
                    model = ARIMA(train, order=(1, 1, 1))
                    model_fit = model.fit()
                    forecast = model_fit.forecast(steps=forecast_horizon)
                    mape = mean_absolute_percentage_error(test, forecast)
                    results[label].append(mape)
            except Exception as e:
                print(f"Skipping {label} for user {user_id} due to error: {e}")

    avg_results = {key: np.mean(val) if val else np.nan for key, val in results.items()}
    print("\nARIMA Forecasting Evaluation Results (Average MAPE):")
    for key, val in avg_results.items():
        print(f"  {key.capitalize()}: {val:.2%}")
    return avg_results




In [ ]:
def run_pipeline():
    print("Loading 400_users_transactions dataset...")
    df = pd.read_csv('400_users_transactions.csv')
    print(f"Dataset loaded with {len(df)} transactions.")

    print("\nStarting feature engineering...")
    features_df = feature_engineering(df)
    features_df.to_csv('400_users_features.csv', index=False)
    print("Features saved to '400_users_features.csv'.")

    print("\nStarting ARIMA evaluation...")
    evaluation_results = evaluate_arima_performance(df, forecast_horizon=3)
    print("Pipeline execution completed.")
    return features_df, evaluation_results